# WiFiSENSE ? ML Baseline Experiment

This notebook compares full and predeclared compact CSI feature sets using Logistic Regression, Random Forest, and RBF-SVM. Validation is Leave-One-Recording-Out for occupancy classes 1 and 2 only. The single class-0 and class-3 recordings remain in every training fold and are never treated as independently validated.

All windows from a held recording are validation rows. No random row/window split is used. Scalers are fitted inside each training-fold pipeline. This is a limited WiCount experiment, not evidence of performance on WiFiSENSE ESP32 hardware or all four occupancy classes.

In [1]:
import hashlib
import json
import platform
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_recall_fscore_support,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits

root = Path(r'G:\My Drive\github\projects\WiFiSANSE')
feature_path = root / 'analysis' / 'processed' / 'features' / 'feature_matrix_50.csv'
recommendation_path = root / 'analysis' / 'processed' / 'features' / 'feature_reduction_recommendation.csv'
archive_path = root / 'analysis' / 'dataset' / 'WiCount' / 'WiCount.zip'
prd_path = root / 'PRD.md'
output_dir = root / 'analysis' / 'outputs' / 'ml_baseline'
output_dir.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

protected_before = {
    'feature_matrix_50.csv': sha256(feature_path),
    'feature_reduction_recommendation.csv': sha256(recommendation_path),
    'WiCount.zip': sha256(archive_path),
    'PRD.md': sha256(prd_path),
}
print('Python:', sys.version.split()[0])
print('scikit-learn:', sklearn.__version__)
print('Platform:', platform.platform())
print('Outputs:', output_dir)


Python: 3.14.7
scikit-learn: 1.9.1
Platform: Windows-11-10.0.26200-SP0
Outputs: G:\My Drive\github\projects\WiFiSANSE\analysis\outputs\ml_baseline


## Data and frozen feature scenarios

The input is the existing 50-sample window feature matrix. Metadata and target are excluded explicitly. The compact columns come directly from the previously prepared `candidate_core` tier; this notebook does not calculate new feature rankings or select features from validation-fold data. The candidate tier is a prior exploratory recommendation, so compact-set results remain a pilot rather than a fully nested feature-selection study.

In [2]:
feature_frame = pd.read_csv(feature_path)
recommendation = pd.read_csv(recommendation_path)
metadata_cols = ['window_id', 'recording_id', 'occupancy', 'window_size', 'start_index', 'end_index', 'row_count']
if feature_frame[metadata_cols].isna().any().any():
    raise ValueError('Required metadata contains missing values.')
if feature_frame['recording_id'].nunique() != 15:
    raise ValueError('Expected exactly 15 recording groups.')
if feature_frame.shape != (3122, 999):
    raise ValueError(f'Unexpected feature matrix shape: {feature_frame.shape}')

full_features = [c for c in feature_frame.columns if c not in metadata_cols]
if len(full_features) != 992:
    raise ValueError(f'Expected 992 numeric features, found {len(full_features)}')
if not all(pd.api.types.is_numeric_dtype(feature_frame[c]) for c in full_features):
    raise TypeError('A non-numeric or metadata column entered the model feature set.')
X_full_all = feature_frame[full_features].to_numpy(dtype=float)
if not np.isfinite(X_full_all).all():
    raise ValueError('Input feature matrix contains NaN or infinite values.')

compact_features = recommendation.loc[recommendation['recommendation_tier'].eq('candidate_core'), 'feature'].tolist()
if len(compact_features) != 211 or len(set(compact_features)) != 211:
    raise ValueError(f'Expected 211 unique frozen candidate_core columns, got {len(compact_features)}')
missing_compact = sorted(set(compact_features) - set(full_features))
if missing_compact:
    raise ValueError(f'Candidate columns missing from feature matrix: {missing_compact}')
if set(metadata_cols) & set(full_features):
    raise ValueError('Metadata/label found in full model feature columns.')

recording_labels = feature_frame.groupby('recording_id')['occupancy'].agg(['nunique', 'first'])
if not recording_labels['nunique'].eq(1).all():
    raise ValueError('A recording_id is associated with more than one occupancy label.')
recordings_by_class = recording_labels.groupby('first').size().to_dict()
expected_recordings = {0: 1, 1: 7, 2: 6, 3: 1}
if recordings_by_class != expected_recordings:
    raise ValueError(f'Unexpected recording counts by class: {recordings_by_class}')
print('Feature matrix:', feature_frame.shape)
print('Full feature count:', len(full_features))
print('Compact candidate count:', len(compact_features))
print('Compact candidate names verified:', compact_features)
print('Recordings per class:', recordings_by_class)
print('Windows per class:', feature_frame['occupancy'].value_counts().sort_index().to_dict())


Feature matrix: (3122, 999)
Full feature count: 992
Compact candidate count: 211
Compact candidate names verified: ['sub_0_mean', 'sub_0_std', 'sub_0_zero_crossing_rate', 'sub_0_first_diff_std', 'sub_1_mean', 'sub_1_std', 'sub_1_zero_crossing_rate', 'sub_1_first_diff_std', 'sub_2_mean', 'sub_2_std', 'sub_2_zero_crossing_rate', 'sub_2_first_diff_std', 'sub_3_mean', 'sub_3_std', 'sub_3_zero_crossing_rate', 'sub_3_first_diff_std', 'sub_4_mean', 'sub_4_std', 'sub_4_zero_crossing_rate', 'sub_4_first_diff_std', 'sub_5_mean', 'sub_5_std', 'sub_5_zero_crossing_rate', 'sub_5_first_diff_std', 'sub_6_mean', 'sub_6_std', 'sub_6_zero_crossing_rate', 'sub_6_first_diff_std', 'sub_7_mean', 'sub_7_std', 'sub_7_zero_crossing_rate', 'sub_7_first_diff_std', 'sub_8_mean', 'sub_8_std', 'sub_8_zero_crossing_rate', 'sub_8_first_diff_std', 'sub_9_mean', 'sub_9_std', 'sub_9_zero_crossing_rate', 'sub_9_first_diff_std', 'sub_10_mean', 'sub_10_std', 'sub_10_zero_crossing_rate', 'sub_10_first_diff_std', 'sub_11_mea

## Leave-One-Recording-Out folds

A fold holds out exactly one recording from class 1 or 2. Class-0 and class-3 recordings are included in training in every fold. This produces 13 folds: seven class-1 holdouts and six class-2 holdouts. Validation in each fold has one true class; per-fold F1 therefore describes only that fold's supported class. Macro-F1 over both evaluated classes is reported on the pooled out-of-fold predictions.

In [3]:
recording_class = recording_labels['first'].astype(int).to_dict()
folds = []
for occupancy in (1, 2):
    class_recordings = sorted([rid for rid, label in recording_class.items() if label == occupancy])
    for rid in class_recordings:
        folds.append({
            'fold_id': f'class_{occupancy}__{rid}',
            'heldout_recording_id': rid,
            'heldout_occupancy': occupancy,
        })
if len(folds) != 13:
    raise ValueError(f'Expected 13 group folds, found {len(folds)}')
if {fold['heldout_occupancy'] for fold in folds} != {1, 2}:
    raise ValueError('Folds do not cover both supported evaluation classes.')

fold_plan_rows = []
for fold in folds:
    heldout = fold['heldout_recording_id']
    train_records = sorted(set(recording_class) - {heldout})
    val_records = [heldout]
    if set(train_records) & set(val_records):
        raise AssertionError('Recording leakage across train/validation groups.')
    if recording_class[heldout] not in (1, 2):
        raise AssertionError('A class-0/3 recording was selected for validation.')
    fold_plan_rows.append({
        **fold,
        'train_recording_count': len(train_records),
        'validation_recording_count': len(val_records),
        'train_recording_ids': '|'.join(train_records),
        'validation_recording_ids': heldout,
        'train_window_count': int((feature_frame['recording_id'] != heldout).sum()),
        'validation_window_count': int((feature_frame['recording_id'] == heldout).sum()),
    })
fold_plan = pd.DataFrame(fold_plan_rows)
fold_plan.to_csv(output_dir / 'fold_plan.csv', index=False)
print('Fold count:', len(fold_plan))
print('Holdout distribution:', fold_plan['heldout_occupancy'].value_counts().sort_index().to_dict())
print('Window counts per fold (train/validation):')
print(fold_plan[['fold_id','train_window_count','validation_window_count']].to_string(index=False))


Fold count: 13
Holdout distribution: {1: 7, 2: 6}
Window counts per fold (train/validation):
                               fold_id  train_window_count  validation_window_count
class_1__2023-12-21_19-27-21-930_104_1                2932                      190
class_1__2023-12-21_19-31-17-706_104_1                2916                      206
class_1__2023-12-21_19-34-42-783_104_1                2914                      208
class_1__2023-12-21_19-37-36-861_104_1                2917                      205
class_1__2023-12-21_19-40-12-110_104_1                2907                      215
class_1__2023-12-21_19-42-28-267_104_1                2898                      224
class_1__2023-12-21_19-45-07-501_104_1                2918                      204
class_2__2023-12-21_19-47-14-669_104_1                2908                      214
class_2__2023-12-21_19-49-37-940_104_1                2911                      211
class_2__2023-12-21_19-54-01-269_104_1                2918         

## Baseline models and training-fold-only preprocessing

Fixed baseline settings (no broad hyperparameter search):

- Logistic Regression: `C=1`, `lbfgs`, `max_iter=2000`, `class_weight='balanced'`; StandardScaler in the fold pipeline.
- Random Forest: 100 trees, `max_features='sqrt'`, `class_weight='balanced_subsample'`, `random_state=42`; no scaler.
- RBF-SVM: `C=1`, `gamma='scale'`, `class_weight='balanced'`; StandardScaler in the fold pipeline.

Class weights are computed by each estimator from that fold's training labels only. The pipelines are newly constructed and fit within each fold; validation data is used only for prediction and scoring.

In [4]:
def build_model(model_name):
    if model_name == 'Logistic Regression':
        estimator = LogisticRegression(
            C=1.0, solver='lbfgs', max_iter=2000,
            class_weight='balanced', random_state=42,
        )
        return Pipeline([('scaler', StandardScaler()), ('model', estimator)])
    if model_name == 'Random Forest':
        return RandomForestClassifier(
            n_estimators=100, max_features='sqrt',
            class_weight='balanced_subsample', random_state=42, n_jobs=1,
        )
    if model_name == 'RBF-SVM':
        estimator = SVC(
            C=1.0, kernel='rbf', gamma='scale',
            class_weight='balanced', random_state=42,
        )
        return Pipeline([('scaler', StandardScaler()), ('model', estimator)])
    raise ValueError(f'Unknown model: {model_name}')

model_names = ['Logistic Regression', 'Random Forest', 'RBF-SVM']
feature_scenarios = {'full': full_features, 'compact': compact_features}
fold_metric_rows = []
oof_rows = []

with threadpool_limits(limits=1):
    for feature_set, columns in feature_scenarios.items():
        X_all = feature_frame[columns].to_numpy(dtype=float)
        y_all = feature_frame['occupancy'].to_numpy(dtype=int)
        group_all = feature_frame['recording_id'].to_numpy()
        for model_name in model_names:
            for fold in folds:
                heldout = fold['heldout_recording_id']
                val_mask = group_all == heldout
                train_mask = ~val_mask
                X_train, X_val = X_all[train_mask], X_all[val_mask]
                y_train, y_val = y_all[train_mask], y_all[val_mask]
                train_groups = set(group_all[train_mask])
                val_groups = set(group_all[val_mask])
                if train_groups & val_groups:
                    raise AssertionError(f'Group leakage in {fold["fold_id"]}')
                if val_groups != {heldout} or len(set(y_val)) != 1:
                    raise AssertionError(f'Invalid validation group or labels in {fold["fold_id"]}')
                if not set([0, 1, 2, 3]).issubset(set(y_train)):
                    raise AssertionError('A training fold is missing one of the four classes.')

                estimator = build_model(model_name)
                fit_start = time.perf_counter()
                estimator.fit(X_train, y_train)
                fit_seconds = time.perf_counter() - fit_start
                if isinstance(estimator, Pipeline):
                    scaler = estimator.named_steps['scaler']
                    if int(np.max(np.asarray(scaler.n_samples_seen_))) != len(y_train):
                        raise AssertionError('Scaler was not fit on exactly the training rows.')
                pred_start = time.perf_counter()
                y_pred = estimator.predict(X_val).astype(int)
                predict_seconds = time.perf_counter() - pred_start

                supported = sorted(set(y_val))
                fold_macro_f1 = f1_score(y_val, y_pred, labels=supported, average='macro', zero_division=0)
                p, r, _, support = precision_recall_fscore_support(
                    y_val, y_pred, labels=[1, 2], average=None, zero_division=0,
                )
                class_metrics = {}
                for idx, cls in enumerate([1, 2]):
                    has_support = bool(support[idx] > 0)
                    class_metrics[f'precision_class_{cls}'] = float(p[idx]) if has_support else np.nan
                    class_metrics[f'recall_class_{cls}'] = float(r[idx]) if has_support else np.nan
                    class_metrics[f'support_class_{cls}'] = int(support[idx])

                fold_metric_rows.append({
                    'feature_set': feature_set,
                    'model': model_name,
                    'fold_id': fold['fold_id'],
                    'heldout_occupancy': int(fold['heldout_occupancy']),
                    'heldout_recording_id': heldout,
                    'train_recording_count': len(train_groups),
                    'validation_recording_count': len(val_groups),
                    'train_window_count': int(len(y_train)),
                    'validation_window_count': int(len(y_val)),
                    'validation_true_class_support': int(len(y_val)),
                    'accuracy': float(accuracy_score(y_val, y_pred)),
                    'macro_f1_present_class_only': float(fold_macro_f1),
                    **class_metrics,
                    'fit_seconds': float(fit_seconds),
                    'predict_seconds': float(predict_seconds),
                })
                validation_rows = feature_frame.loc[val_mask, metadata_cols].reset_index(drop=True)
                for row_index, pred in enumerate(y_pred):
                    oof_rows.append({
                        'feature_set': feature_set,
                        'model': model_name,
                        'fold_id': fold['fold_id'],
                        'heldout_recording_id': heldout,
                        'window_id': validation_rows.loc[row_index, 'window_id'],
                        'recording_id': validation_rows.loc[row_index, 'recording_id'],
                        'y_true': int(validation_rows.loc[row_index, 'occupancy']),
                        'y_pred': int(pred),
                    })

fold_metrics = pd.DataFrame(fold_metric_rows)
oof_predictions = pd.DataFrame(oof_rows)
expected_oof_rows = (1452 + 1268) * len(feature_scenarios) * len(model_names)
if fold_metrics.shape[0] != 78:
    raise AssertionError(f'Expected 78 fold-model-feature rows, got {fold_metrics.shape[0]}')
if len(oof_predictions) != expected_oof_rows:
    raise AssertionError(f'Unexpected OOF row count: {len(oof_predictions)}')
if not oof_predictions['recording_id'].eq(oof_predictions['heldout_recording_id']).all():
    raise AssertionError('An OOF prediction is not from its held-out recording.')
if not oof_predictions['y_true'].isin([1, 2]).all():
    raise AssertionError('Class 0 or 3 unexpectedly appears in true validation labels.')
fold_metrics.to_csv(output_dir / 'fold_metrics.csv', index=False)
oof_predictions.to_csv(output_dir / 'oof_predictions.csv', index=False)
print('Completed model fits:', len(fold_metrics))
print('OOF prediction rows:', len(oof_predictions))
print('OOF rows per scenario:', oof_predictions.groupby(['feature_set','model']).size().to_dict())
print('Mean fit time per fold (s):')
print(fold_metrics.groupby(['feature_set','model'])['fit_seconds'].mean().round(4).to_string())


Completed model fits: 78
OOF prediction rows: 16320
OOF rows per scenario: {('compact', 'Logistic Regression'): 2720, ('compact', 'RBF-SVM'): 2720, ('compact', 'Random Forest'): 2720, ('full', 'Logistic Regression'): 2720, ('full', 'RBF-SVM'): 2720, ('full', 'Random Forest'): 2720}
Mean fit time per fold (s):
feature_set  model              
compact      Logistic Regression    1.6711
             RBF-SVM                0.5914
             Random Forest          3.8002
full         Logistic Regression    5.7514
             RBF-SVM                2.7433
             Random Forest          8.7638


## Pooled out-of-fold metrics and error review

The primary pooled comparison uses all held-out windows from classes 1 and 2. Accuracy and macro-F1 are reported only on this evaluation scope. Per-class metrics for classes 0 and 3 are intentionally omitted because those labels have no true validation samples.

In [5]:
comparison_rows = []
error_rows = []
recording_rows = []
for (feature_set, model_name), subset in oof_predictions.groupby(['feature_set', 'model'], sort=False):
    y_true = subset['y_true'].to_numpy(dtype=int)
    y_pred = subset['y_pred'].to_numpy(dtype=int)
    p, r, f, support = precision_recall_fscore_support(
        y_true, y_pred, labels=[1, 2], average=None, zero_division=0,
    )
    folds_subset = fold_metrics[(fold_metrics['feature_set'] == feature_set) & (fold_metrics['model'] == model_name)]
    comparison_rows.append({
        'feature_set': feature_set,
        'model': model_name,
        'evaluated_recordings': int(subset['recording_id'].nunique()),
        'evaluated_windows': int(len(subset)),
        'true_support_class_1': int(support[0]),
        'precision_class_1': float(p[0]),
        'recall_class_1': float(r[0]),
        'f1_class_1': float(f[0]),
        'true_support_class_2': int(support[1]),
        'precision_class_2': float(p[1]),
        'recall_class_2': float(r[1]),
        'f1_class_2': float(f[1]),
        'accuracy_oof': float(accuracy_score(y_true, y_pred)),
        'macro_f1_classes_1_2_oof': float(f1_score(y_true, y_pred, labels=[1, 2], average='macro', zero_division=0)),
        'mean_recording_accuracy': float(folds_subset['accuracy'].mean()),
        'std_recording_accuracy': float(folds_subset['accuracy'].std(ddof=1)),
        'mean_fit_seconds_per_fold': float(folds_subset['fit_seconds'].mean()),
        'total_fit_seconds': float(folds_subset['fit_seconds'].sum()),
        'misclassified_windows': int(np.sum(y_true != y_pred)),
        'predicted_as_class_0': int(np.sum(y_pred == 0)),
        'predicted_as_class_3': int(np.sum(y_pred == 3)),
    })
    for true_cls in (1, 2):
        for pred_cls in (0, 1, 2, 3):
            error_rows.append({
                'feature_set': feature_set,
                'model': model_name,
                'y_true': true_cls,
                'y_pred': pred_cls,
                'count': int(((y_true == true_cls) & (y_pred == pred_cls)).sum()),
            })
    by_recording = subset.groupby(['heldout_recording_id','y_true'], as_index=False).agg(
        windows=('window_id','size'),
        accuracy=('y_true', lambda values: float(np.mean(values.to_numpy() == subset.loc[values.index, 'y_pred'].to_numpy()))),
    )
    by_recording.insert(0, 'model', model_name)
    by_recording.insert(0, 'feature_set', feature_set)
    recording_rows.extend(by_recording.to_dict('records'))

model_comparison = pd.DataFrame(comparison_rows)
error_analysis = pd.DataFrame(error_rows)
recording_oof = pd.DataFrame(recording_rows)
model_comparison.to_csv(output_dir / 'model_comparison.csv', index=False)
error_analysis.to_csv(output_dir / 'error_analysis.csv', index=False)
recording_oof.to_csv(output_dir / 'recording_oof_metrics.csv', index=False)
print(model_comparison[['feature_set','model','accuracy_oof','macro_f1_classes_1_2_oof','precision_class_1','recall_class_1','precision_class_2','recall_class_2','mean_recording_accuracy','total_fit_seconds']].round(4).to_string(index=False))
print('Misclassification counts by true/predicted class:')
print(error_analysis[error_analysis['y_true'] != error_analysis['y_pred']].to_string(index=False))


feature_set               model  accuracy_oof  macro_f1_classes_1_2_oof  precision_class_1  recall_class_1  precision_class_2  recall_class_2  mean_recording_accuracy  total_fit_seconds
       full Logistic Regression        0.7989                    0.8291             0.8635          0.8753             0.8852          0.7114                   0.8011            74.7685
       full       Random Forest        0.8551                    0.8582             0.8540          0.9022             0.8804          0.8013                   0.8566           113.9295
       full             RBF-SVM        0.8327                    0.8587             0.8809          0.9167             0.9220          0.7366                   0.8346            35.6635
    compact Logistic Regression        0.7886                    0.8253             0.8712          0.8760             0.8917          0.6885                   0.7907            21.7245
    compact       Random Forest        0.8596                    0.863

In [6]:
# Four-class confusion matrix: rows 0 and 3 have zero true validation support by design.
class_labels = [0, 1, 2, 3]
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, ((feature_set, model_name), subset) in zip(axes.flat, oof_predictions.groupby(['feature_set','model'], sort=False)):
    cm = confusion_matrix(subset['y_true'], subset['y_pred'], labels=class_labels)
    image = ax.imshow(cm, cmap='Blues')
    for row in range(4):
        for col in range(4):
            ax.text(col, row, str(cm[row, col]), ha='center', va='center', color='black', fontsize=9)
    support = cm.sum(axis=1)
    ax.set_xticks(range(4), labels=class_labels)
    ax.set_yticks(range(4), labels=[f'{label} (n={support[label]})' for label in class_labels])
    ax.set_xlabel('Predicted occupancy')
    ax.set_ylabel('True occupancy / support')
    ax.set_title(f'{feature_set}: {model_name}')
fig.suptitle('Pooled OOF confusion matrices; classes 0 and 3 have no true validation samples', y=1.01)
fig.tight_layout()
fig.savefig(output_dir / 'confusion_matrix.png', dpi=180, bbox_inches='tight')
plt.close(fig)

# Compare pooled accuracy and macro-F1 for classes 1 and 2.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
metrics_to_plot = [('accuracy_oof','OOF accuracy'), ('macro_f1_classes_1_2_oof','Macro-F1 (classes 1 and 2)')]
model_order = model_names
x = np.arange(len(model_order))
width = 0.34
for ax, (metric, title) in zip(axes, metrics_to_plot):
    full = model_comparison[model_comparison['feature_set']=='full'].set_index('model').reindex(model_order)[metric]
    compact = model_comparison[model_comparison['feature_set']=='compact'].set_index('model').reindex(model_order)[metric]
    ax.bar(x - width/2, full, width, label='Full (992)', color='steelblue')
    ax.bar(x + width/2, compact, width, label='Compact candidate (211)', color='darkorange')
    ax.set_xticks(x, labels=model_order, rotation=15)
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.set_ylabel('Score')
    ax.legend()
fig.suptitle('OOF comparison restricted to true validation classes 1 and 2')
fig.tight_layout()
fig.savefig(output_dir / 'feature_comparison.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved confusion matrix and feature comparison plots.')


Saved confusion matrix and feature comparison plots.


## Reproducibility and final quality checks

The manifest records package versions, feature names, model settings, group-fold counts, and protected-input checksums. Predictions in the OOF table must refer only to their held-out recording. Validation recordings must be unique across folds for their occupancy class; no recording may occur in train and validation in one fold.

In [7]:
expected_heldouts = {rid for rid, label in recording_class.items() if label in (1, 2)}
for (feature_set, model_name), subset in oof_predictions.groupby(['feature_set','model']):
    if set(subset['heldout_recording_id'].unique()) != expected_heldouts:
        raise AssertionError(f'OOF holdouts incomplete for {feature_set}/{model_name}')
    if subset.groupby('window_id').size().ne(1).any():
        raise AssertionError(f'A window received more than one OOF prediction for {feature_set}/{model_name}')
    if subset['recording_id'].nunique() != 13:
        raise AssertionError('Expected 13 distinct evaluated recordings per model/scenario.')
if set(oof_predictions['y_true'].unique()) != {1, 2}:
    raise AssertionError('OOF true labels should contain only classes 1 and 2.')
if (fold_metrics['train_recording_count'] != 14).any() or (fold_metrics['validation_recording_count'] != 1).any():
    raise AssertionError('Unexpected group counts in one or more folds.')

protected_after = {
    'feature_matrix_50.csv': sha256(feature_path),
    'feature_reduction_recommendation.csv': sha256(recommendation_path),
    'WiCount.zip': sha256(archive_path),
    'PRD.md': sha256(prd_path),
}
if protected_before != protected_after:
    raise AssertionError('A protected dataset, feature baseline, recommendation, or PRD changed.')
manifest = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'scikit_learn': sklearn.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'random_state': 42,
    'fold_count': len(folds),
    'folds_by_heldout_class': fold_plan['heldout_occupancy'].value_counts().sort_index().to_dict(),
    'feature_counts': {'full': len(full_features), 'compact': len(compact_features)},
    'models': model_names,
    'model_settings': {
        'logistic_regression': 'C=1.0, solver=lbfgs, max_iter=2000, class_weight=balanced, random_state=42; StandardScaler in fold pipeline',
        'random_forest': 'n_estimators=100, max_features=sqrt, class_weight=balanced_subsample, random_state=42, n_jobs=1; no scaler',
        'rbf_svm': 'C=1.0, gamma=scale, class_weight=balanced; StandardScaler in fold pipeline',
    },
    'validation_scope': 'OOF recording-level holdout for classes 1 and 2 only; classes 0 and 3 always train-only',
    'protected_sha256_before_and_after': protected_after,
    'model_training_performed': True,
}
with open(output_dir / 'run_manifest.json', 'w', encoding='utf-8') as stream:
    json.dump(manifest, stream, indent=2)
print('Validation groups are disjoint within every fold.')
print('Scaler fitted only on training rows in each scaled pipeline.')
print('All OOF predictions belong to their held-out recording.')
print('Protected dataset, feature files, and PRD unchanged:', protected_before == protected_after)
print('Run manifest:', output_dir / 'run_manifest.json')
print('No predictions are reported as validation for classes 0 and 3.')


Validation groups are disjoint within every fold.
Scaler fitted only on training rows in each scaled pipeline.
All OOF predictions belong to their held-out recording.
Protected dataset, feature files, and PRD unchanged: True
Run manifest: G:\My Drive\github\projects\WiFiSANSE\analysis\outputs\ml_baseline\run_manifest.json
No predictions are reported as validation for classes 0 and 3.
